<a href="https://colab.research.google.com/github/Vx-N3AVH-3LVS-xV/Harvard-CS50P/blob/main/python0_4chemistry_struturure_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
name = "Vinh"
subject = "Computer Science"
years = str(13)
print("Hello, I am "+ name)
print("I love "+ subject)
print("In "+ years+ " years, I want to build my own research project!")
#str() converts data into a string of characters , to insert a const or a var into a string , must convert into a string.
#3x+1 counter
x = int(input("Define starting value             "))
while x != 4 :
  print(int(x))
  if x%2 == 1 :
    x= 3*x+1
  else:
    x= x/2

Hello, I am Vinh
I love Computer Science
In 13 years, I want to build my own research project!
Define starting value             13
13
40
20
10
5
16
8


In [ ]:

import re
import math
from itertools import product

VALENCY_MAP_STRING = "H:[1,-1]:1|He:[]:2|Li:[1]:3|Be:[2]:4|B:[3]:5|C:[4,2,-4]:6|N:[3,5,-3]:7|O:[-2,2]:8|F:[-1]:9|Ne:[]:10|Na:[1]:11|Mg:[2]:12|Al:[3]:13|Si:[4,-4]:14|P:[3,5,-3]:15|S:[2,4,6,-2]:16|Cl:[1,3,5,7,-1]:17|Ar:[]:18|K:[1]:19|Ca:[2]:20|Sc:[3]:21|Ti:[2,3,4]:22|V:[2,3,4,5]:23|Cr:[2,3,6]:24|Mn:[2,3,4,6,7]:25|Fe:[2,3]:26|Cu:[1,2]:29|Zn:[2]:30|Br:[1,3,5,7,-1]:35|Ag:[1]:47|I:[1,3,5,7,-1]:53|Au:[1,3]:79|Pb:[2,4]:82"

VALENCY_MAP = {
    parts[0]: {
        "name": "",
        "valency": [int(v.strip()) for v in parts[1].strip("[]").split(",") if v.strip()] if parts[1] != "[]" else [],
        "protons": int(parts[2])
    }
    for item in VALENCY_MAP_STRING.split("|")
    for parts in [item.split(":")]
}

class Compound:
    def __init__(self, formula_string):
        self.raw_formula = formula_string
        self.composition = {}
        self.is_valid_valency = False
        self.is_acid = False
        self.is_base = False
        self.is_stable = True
        self.is_simplified = True
        self.suggested_formula = None
        self.stability_notes = "Stable under standard ambient conditions."
        self._analyze_compound()

    def _parse_formula(self):
        pattern = r"([A-Z][a-z]*)(\d*)"
        matches = re.findall(pattern, self.raw_formula)

        if not matches:
            raise ValueError("Formula string structure could not be parsed.")

        parsed = {}
        total_len = 0
        previous_symbol = None

        for symbol, count in matches:
            if symbol not in VALENCY_MAP:
                raise ValueError(f"Unknown or unsupported element symbol: '{symbol}'")

            if symbol == previous_symbol:
                raise ValueError(f"Invalid consecutive duplication of element symbol '{symbol}'. Use numeric subscripts (e.g., 'Na2') instead.")

            current_count = int(count) if count else 1

            parsed[symbol] = parsed.get(symbol, 0) + current_count
            total_len += len(symbol) + len(count)
            previous_symbol = symbol

        if total_len != len(self.raw_formula):
            raise ValueError("Formula contains invalid characters or incorrect casing.")
        return parsed

    def _check_simplification(self):
        counts = list(self.composition.values())
        if len(counts) < 2:
            return

        common_divisor = math.gcd(*counts)

        if common_divisor > 1:
            self.is_simplified = False
            self.is_stable = False
            self.stability_notes = f"Duplicate/Unsimplified ratio detected (Factor of {common_divisor})."

            suggestion_parts = []
            for symbol, count in self.composition.items():
                reduced_count = count // common_divisor
                count_str = str(reduced_count) if reduced_count > 1 else ""
                suggestion_parts.append(f"{symbol}{count_str}")
            self.suggested_formula = "".join(suggestion_parts)

    def _check_valency_balance(self):
        try:
            self.composition = self._parse_formula()
        except ValueError:
            raise

        self._check_simplification()
        if not self.is_simplified:
            return False

        symbols = list(self.composition.keys())
        valency_choices = [VALENCY_MAP[sym]["valency"] for sym in symbols]

        for combination in product(*valency_choices):
            net_charge = sum(val * self.composition[sym] for sym, val in zip(symbols, combination))
            if net_charge == 0:
                return True
        return False

    def _analyze_compound(self):
        valid_complex_overrides = ["NH4OH", "NaOH", "KOH", "Ca(OH)2", "HClO2", "NaClO2"]

        if self._check_valency_balance() or self.raw_formula in valid_complex_overrides:
            self.is_valid_valency = True
        else:
            self.is_valid_valency = False
            if self.is_simplified:
                self.is_stable = False
                self.stability_notes = "Valency calculations fail to yield a neutral chemical structure."
            return

        is_ammonium_hydroxide = self.raw_formula == "NH4OH"
        if is_ammonium_hydroxide or "NH4" in self.raw_formula:
            self.is_stable = False
            self.stability_notes = "Unstable! Exists only as a transient aqueous solution; rapidly degrades releasing NH3 gas."
        elif self.raw_formula == "HO":
            self.is_stable = False
            self.stability_notes = "Unstable radical (Hydroxyl)."

        if self.raw_formula.startswith("H") and self.raw_formula not in ["H2O", "H2O2"]:
            self.is_acid = True
        elif any(metal in self.raw_formula for metal in ["Na", "K", "Li", "Mg", "Ca"]) and ("O" in self.raw_formula or "OH" in self.raw_formula or "ClO2" in self.raw_formula):
            self.is_base = self.raw_formula.endswith("OH")
        elif is_ammonium_hydroxide:
            self.is_base = True

    def display_report(self):
        print(f"\nChemical Profile for: [{self.raw_formula}]")
        print(f"  - Parsed Map         : {self.composition}")
        print(f"  - Lowest Ratio (Emp) : {self.is_simplified}")
        print(f"  - Structurally Valid : {self.is_valid_valency}")
        print(f"  - Thermodynamic Safe : {self.is_stable}")
        print(f"  - Stability Diagnosis: {self.stability_notes}")
        if self.suggested_formula:
            print(f"  - Did you mean?      : {self.suggested_formula}")
        print("=" * 50)

def main():
    print("Type 'exit' or 'quit' to close the program.\n")
    while True:
        user_input = input("Enter chemical formula: ").strip()
        if user_input.lower() in ['exit', 'quit', '']:
            break
        try:
            chemical_obj = Compound(user_input)
            chemical_obj.display_report()
        except ValueError as e:
            print(f"Input Error: {e}")
            print("=" * 50)

if __name__ == "__main__":
    command = input("Code name: ").strip()
    if command == "Identify":
        main()


Code name: Identify
Type 'exit' or 'quit' to close the program.

Enter chemical formula: AgNO3

Chemical Profile for: [AgNO3]
  - Parsed Map         : {'Ag': 1, 'N': 1, 'O': 3}
  - Lowest Ratio (Emp) : True
  - Structurally Valid : True
  - Thermodynamic Safe : True
  - Stability Diagnosis: Stable under standard ambient conditions.
Enter chemical formula: KNaMgO3

Chemical Profile for: [KNaMgO3]
  - Parsed Map         : {'K': 1, 'Na': 1, 'Mg': 1, 'O': 3}
  - Lowest Ratio (Emp) : True
  - Structurally Valid : False
  - Thermodynamic Safe : False
  - Stability Diagnosis: Valency calculations fail to yield a neutral chemical structure.
Enter chemical formula: KNaMgMn2O9

Chemical Profile for: [KNaMgMn2O9]
  - Parsed Map         : {'K': 1, 'Na': 1, 'Mg': 1, 'Mn': 2, 'O': 9}
  - Lowest Ratio (Emp) : True
  - Structurally Valid : True
  - Thermodynamic Safe : True
  - Stability Diagnosis: Stable under standard ambient conditions.
Enter chemical formula: AgH2PO3

Chemical Profile for: [AgH2PO